In [9]:
from pandas.core.interchange.dataframe_protocol import DataFrame

from _bootstrap import setup
from src.vasijas.reporting import ProcessingReport, ProcessingResult

import cv2
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from enum import StrEnum
import os
import pandas as pd

In [10]:
class OrientacionCorte(StrEnum):
    VERTICAL = "vertical"
    HORIZONTAL = "horizontal"

In [11]:
def obtener_o_crear_directorio(ruta_base: Path, carpeta_destino: str):
    ruta_completa = ruta_base / carpeta_destino

    ruta_completa.mkdir(parents=True, exist_ok=True)

    if not ruta_completa.exists():
        raise RuntimeError("No se pudo crear el directorio")

    return ruta_completa


# Funciones
def obtener_nombre_carpeta_clase(valor_clase: str | int | float):
    """
    Devuelve el nombre de la carpeta según la 'Clase Ana':
    - Si hay número → 'clase01', 'clase02', ... 'clase11', etc.
    - Si está vacío/NaN → 'sin clase'
    """
    if pd.isna(valor_clase) or str(valor_clase).strip() == "":
        return "sin_clase"
    try:
        numero = int(float(valor_clase))  # maneja '11.0' como 11
        return f"clase{numero:02d}"
    except ValueError:
        return "sin_clase"



In [12]:
def obtener_rutas_perfiles(
        ruta_origen: Path, clase: int | str, extension: str
) -> list[Path]:
    nombre_carpeta = obtener_nombre_carpeta_clase(clase)
    carpeta = ruta_origen / nombre_carpeta

    return list(carpeta.glob(f"*.{extension}"))

In [13]:
def determinar_orientacion_corte(clase: str) -> OrientacionCorte:
    if clase in range(1, 10):
        return OrientacionCorte.VERTICAL
    return OrientacionCorte.HORIZONTAL


def crear_ruta_archivo(ruta_base: Path, nombre_archivo: str, sufijo: str):
    return os.path.join(ruta_base, f"{nombre_archivo}{sufijo}.png")

In [14]:
def mostrar_imagen(imagen):
    plt.imshow(imagen, cmap=plt.cm.gray)
    plt.axis("on")
    plt.figure()


def obtener_nombre_archivo(ruta: Path) -> str:
    return Path(ruta).stem


def guardar_imagen(ruta: Path, imagen):
    cv2.imwrite(ruta, imagen)


def leer_imagen(ruta_imagen: str):
    img = cv2.imread(ruta_imagen, cv2.IMREAD_GRAYSCALE)
    return img


def cortar_imagen(img, orientacion_corte: OrientacionCorte):
    alto, ancho = img.shape

    if orientacion_corte == OrientacionCorte.VERTICAL:
        tope_slice = (slice(None, alto // 2), slice(None))
        base_slice = (slice(alto // 2, None), slice(None))
    else:
        tope_slice = (slice(None), slice(None, ancho // 2))
        base_slice = (slice(None), slice(ancho // 2, None))

    fondo_tope = np.full_like(img, 255)
    fondo_base = np.full_like(img, 255)

    fondo_tope[tope_slice] = img[tope_slice]
    fondo_base[base_slice] = img[base_slice]

    return (fondo_tope, fondo_base)



In [15]:
def procesar_perfil(
        ruta_perfil: Path,
        orientacion: OrientacionCorte,
        ruta_destino_clase: Path
) -> ProcessingResult:
    report = ProcessingReport()

    try:
        img = leer_imagen(ruta_perfil)
        tope, base = cortar_imagen(img, orientacion)
        nombre_original = obtener_nombre_archivo(ruta_perfil)

        ruta_tope = crear_ruta_archivo(ruta_destino_clase, nombre_original, "_tope")
        ruta_base = crear_ruta_archivo(ruta_destino_clase, nombre_original, "_base")

        guardar_imagen(ruta=ruta_tope, imagen=tope)
        guardar_imagen(ruta=ruta_base, imagen=base)

        df = pd.DataFrame([
            {
                "archivo_origen": ruta_perfil.name,
                "archivo_generado": ruta_tope.name,
                "mitad": "tope",
            },
            {
                "archivo_origen": ruta_perfil.name,
                "archivo_generado": ruta_base.name,
                "mitad": "base",
            },
        ])

        report.total_perfiles = 1
        report.total_imagenes_generadas = 2

        return ProcessingResult(df=df, report=report)

    except Exception as e:
        report.registrar_error(f"Error procesando {ruta_perfil}: {e}")
        return ProcessingResult(df=pd.DataFrame(), report=report)


def procesar_clase(
        ruta_origen: Path,
        ruta_destino: Path,
        clase: int | str
) -> ProcessingResult:
    report = ProcessingReport()
    dfs: list[pd.DataFrame] = []

    perfiles = obtener_rutas_perfiles(ruta_origen, clase, extension="png")
    if not perfiles:
        return ProcessingResult(df=pd.DataFrame(), report=report)

    orientacion = determinar_orientacion_corte(clase)
    ruta_destino_clase = obtener_o_crear_directorio(
        ruta_destino,
        obtener_nombre_carpeta_clase(clase)
    )

    report.total_clases = 1

    for ruta_perfil in perfiles:
        resultado = procesar_perfil(ruta_perfil, orientacion, ruta_destino_clase)
        dfs.append(resultado.df)

        report.total_perfiles += resultado.report.total_perfiles
        report.total_imagenes_generadas += resultado.report.total_imagenes_generadas
        report.errores.extend(resultado.report.errores)

    df_final = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
    return ProcessingResult(df=df_final, report=report)


def procesar_imagenes(
        ruta_origen: Path,
        ruta_destino: Path,
        clases: list[int | str]
) -> ProcessingResult:
    report = ProcessingReport()
    dfs: list[pd.DataFrame] = []

    total_clases = 0
    total_perfiles = 0
    total_imagenes_generadas = 0

    for clase in clases:
        resultado = procesar_clase(ruta_origen, ruta_destino, clase)
        dfs.append(resultado.df)

        total_clases += resultado.report.total_clases
        total_perfiles += resultado.report.total_perfiles
        total_imagenes_generadas += resultado.report.total_imagenes_generadas
        errores.extend(resultado.report.errores)

    df_final = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
    return ProcessingResult(df=df_final, report=report)


In [16]:
from _bootstrap import setup

clases_a_clasificar = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 'sin_clase']
PATHS = setup()

report = ProcessingReport()
df = pd.DataFrame()

resultado = procesar_imagenes(PATHS.DATA_CLASIFICADOS, PATHS.DATA_MITADES, clases_a_clasificar)

df = resultado.df
report = resultado.report


TypeError: cannot unpack non-iterable NoneType object